In [1]:
!pip install "pyautogen[retrievechat]==0.2.35" -qqq

## 예제 15.2 OpenAI API 키 설정

In [3]:
## 예제 15.2 OpenAI API 키 설정

import json
import os

# 여기에 본인의 OpenAI API 키를 입력하세요
openai_api_key = os.getenv("OPENAI_API_KEY")

with open('OAI_CONFIG_LIST.json', 'w') as f:
    config_list = [
        {
            "model": "gpt-4o-mini",
            "api_key": openai_api_key,
        },
        {
            "model": "gpt-4o",
            "api_key": openai_api_key,
        },
        {
            "model": "gpt-image-2",  # 최신 최상위 이미지 모델
            "api_key": openai_api_key,
        },
        {
            "model": "gpt-image-1.5", # 기존 실습 모델
            "api_key": openai_api_key,
        }
    ]
    json.dump(config_list, f)

## 예제 15.3 에이전트에 사용할 설정 불러오기

In [4]:
## 예제 15.3 에이전트에 사용할 설정 불러오기

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # Colab 시스템 경고 무시

import autogen
import logging

# Set the logging level for autogen.oai.client to ERROR to suppress warnings
logging.getLogger('autogen.oai.client').setLevel(logging.ERROR)

config_list = autogen.config_list_from_json(
    "OAI_CONFIG_LIST.json",
    file_location=".",
    filter_dict={
        "model": ["gpt-4o-mini"],  # 토큰 에러를 방지하기 위해 가볍고 강력한 gpt-4o-mini 지정
    },
)

llm_config = {
    "config_list": config_list,
    "temperature": 0,
}

## 예제 15.4 AutoGen의 핵심 구성요소인 UserProxyAgent와 AssistantAgent

In [5]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent("assistant", llm_config=llm_config)
user_proxy = UserProxyAgent("user_proxy",
  is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
  human_input_mode="NEVER",
  code_execution_config={"work_dir": "coding", "use_docker": False})

## 예제 15.5 삼성전자의 3개월 주식 가격 그래프를 그리는 작업 실행

In [6]:
user_proxy.initiate_chat(assistant, message="""
삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장해줘.
plotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.
값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.
이미지 비율은 보기 좋게 적절히 설정해줘.
""")

user_proxy (to assistant):


삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장해줘.
plotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.
값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.
이미지 비율은 보기 좋게 적절히 설정해줘.


--------------------------------------------------------------------------------
assistant (to user_proxy):

먼저, 삼성전자의 지난 3개월 주식 가격 데이터를 가져와야 합니다. 이를 위해 `yfinance` 라이브러리를 사용하여 데이터를 다운로드하고, `plotly` 라이브러리를 사용하여 그래프를 그리겠습니다. 

아래는 이 작업을 수행하는 Python 코드입니다. 이 코드를 실행하면 삼성전자의 주식 가격 그래프가 생성되고, 지정된 형식으로 저장됩니다.

```python
# filename: plot_samsung_stock.py
import yfinance as yf
import plotly.graph_objects as go
from datetime import datetime, timedelta

# 삼성전자 주식 데이터 다운로드
end_date = datetime.now()
start_date = end_date - timedelta(days=90)
samsung_stock = yf.download('005930.KS', start=start_date, end=end_date)

# 그래프 생성
fig = go.Figure()

# 주식 가격 선 그래프 추가
fig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock['Close'], mode='lines', name='Close Price'))

# 그래프 아래를 투명한 녹색으로 채우기
fig.add_trace(go.

ChatResult(chat_id=None, chat_history=[{'content': '\n삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장해줘.\nplotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.\n값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.\n이미지 비율은 보기 좋게 적절히 설정해줘.\n', 'role': 'assistant', 'name': 'user_proxy'}, {'content': '먼저, 삼성전자의 지난 3개월 주식 가격 데이터를 가져와야 합니다. 이를 위해 `yfinance` 라이브러리를 사용하여 데이터를 다운로드하고, `plotly` 라이브러리를 사용하여 그래프를 그리겠습니다. \n\n아래는 이 작업을 수행하는 Python 코드입니다. 이 코드를 실행하면 삼성전자의 주식 가격 그래프가 생성되고, 지정된 형식으로 저장됩니다.\n\n```python\n# filename: plot_samsung_stock.py\nimport yfinance as yf\nimport plotly.graph_objects as go\nfrom datetime import datetime, timedelta\n\n# 삼성전자 주식 데이터 다운로드\nend_date = datetime.now()\nstart_date = end_date - timedelta(days=90)\nsamsung_stock = yf.download(\'005930.KS\', start=start_date, end=end_date)\n\n# 그래프 생성\nfig = go.Figure()\n\n# 주식 가격 선 그래프 추가\nfig.add_trace(go.Scatter(x=samsung_stock.index, y=samsung_stock[\'Close\'], mode=\'lines\', name=\'Close Price\'))\n\n# 그래프 아래를 투명한 녹색으로 채우기\nfig.add_t

## 예제 15.6 RAG 에이전트 클래스를 사용한 작업 실행

In [7]:
# 예제 15.6 RAG 에이전트 클래스를 사용한 작업 실행

import autogen
from autogen.agentchat.contrib.retrieve_assistant_agent import RetrieveAssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

assistant = RetrieveAssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config=llm_config,
)

ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    retrieve_config={
        "task": "qa",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
        "collection_name": "default-sentence-transformers"
    },
)

assistant.reset()
ragproxyagent.initiate_chat(assistant, problem="AutoGen이 뭐야?")

# assistant (to ragproxyagent):
# AutoGen은 여러 에이전트가 상호 대화하여 작업을 해결할 수 있는 LLM(Large Language Model) 애플리케이션 개발을 가능하게 하는 프레임워크입니다. AutoGen 에이전트는 사용자 정의 가능하며, 대화 가능하고, 인간 참여를 원활하게 허용합니다. LLM, 인간 입력, 도구의 조합을 사용하는 다양한 모드에서 작동할 수 있습니다.

/tmp/ipykernel_2921/2724292186.py:7: DeprecationWarning: The RetrieveAssistantAgent is deprecated. Please use the AssistantAgent instead.
  assistant = RetrieveAssistantAgent(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ragproxyagent (to assistant):

AutoGen이 뭐지?

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

AutoGen은 일반적으로 자동 생성(Auto Generation)과 관련된 기술이나 도구를 의미합니다. 다양한 분야에서 사용될 수 있으며, 특히 소프트웨어 개발, 문서 작성, 데이터 생성 등에서 활용됩니다. 예를 들어, 코드 자동 생성 도구는 개발자가 수동으로 작성해야 하는 코드를 자동으로 생성해 주어 개발 효율성을 높이는 데 도움을 줍니다.

특정한 맥락이나 분야에 따라 AutoGen의 의미가 달라질 수 있으므로, 더 구체적인 정보나 사용 사례를 제공해 주시면 더 자세한 설명을 드릴 수 있습니다.

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.


ChatResult(chat_id=None, chat_history=[{'content': 'AutoGen이 뭐지?', 'role': 'assistant', 'name': 'ragproxyagent'}, {'content': 'AutoGen은 일반적으로 자동 생성(Auto Generation)과 관련된 기술이나 도구를 의미합니다. 다양한 분야에서 사용될 수 있으며, 특히 소프트웨어 개발, 문서 작성, 데이터 생성 등에서 활용됩니다. 예를 들어, 코드 자동 생성 도구는 개발자가 수동으로 작성해야 하는 코드를 자동으로 생성해 주어 개발 효율성을 높이는 데 도움을 줍니다.\n\n특정한 맥락이나 분야에 따라 AutoGen의 의미가 달라질 수 있으므로, 더 구체적인 정보나 사용 사례를 제공해 주시면 더 자세한 설명을 드릴 수 있습니다.', 'role': 'user', 'name': 'assistant'}], summary='AutoGen은 일반적으로 자동 생성(Auto Generation)과 관련된 기술이나 도구를 의미합니다. 다양한 분야에서 사용될 수 있으며, 특히 소프트웨어 개발, 문서 작성, 데이터 생성 등에서 활용됩니다. 예를 들어, 코드 자동 생성 도구는 개발자가 수동으로 작성해야 하는 코드를 자동으로 생성해 주어 개발 효율성을 높이는 데 도움을 줍니다.\n\n특정한 맥락이나 분야에 따라 AutoGen의 의미가 달라질 수 있으므로, 더 구체적인 정보나 사용 사례를 제공해 주시면 더 자세한 설명을 드릴 수 있습니다.', cost={'usage_including_cached_inference': {'total_cost': 8.264999999999999e-05, 'gpt-4o-mini-2024-07-18': {'cost': 8.264999999999999e-05, 'prompt_tokens': 27, 'completion_tokens': 131, 'total_tokens': 158}}, 'usage_excluding_cached_inference': {'total

## 예제 15.7 외부 정보를 활용하지 못하는 기본 에이전트의 답변

In [8]:
# 예제 15.7 외부 정보를 활용하지 못하는 기본 에이전트의 답변

assistant.reset()
userproxyagent = autogen.UserProxyAgent(
    name="userproxyagent",
)
userproxyagent.initiate_chat(assistant, message="Autogen이 뭐야?")

# assistant (to userproxyagent):
# "Autogen"은 자동 생성을 의미하는 용어로, 주로 컴퓨터 프로그래밍에서 사용됩니다. 이는 코드, 문서, 또는 다른 데이터를 자동으로 생성하는 프로세스를 가리킵니다. 이는 반복적인 작업을 줄이고, 효율성을 높이며, 오류를 줄일 수 있습니다. 특정 컨텍스트에 따라 "Autogen"의 정확한 의미는 다를 수 있습니다.

userproxyagent (to assistant):

Autogen이 뭐야?

--------------------------------------------------------------------------------
assistant (to userproxyagent):

Autogen은 일반적으로 "자동 생성"을 의미하는 용어로, 다양한 분야에서 사용될 수 있습니다. 예를 들어, 소프트웨어 개발에서는 코드, 문서, 테스트 케이스 등을 자동으로 생성하는 도구나 프로세스를 지칭할 수 있습니다. 

특히, 게임 개발이나 그래픽 디자인에서는 환경, 캐릭터, 또는 기타 자산을 자동으로 생성하는 기술을 의미할 수 있습니다. 이러한 자동 생성 기술은 개발자의 작업을 줄이고, 더 많은 콘텐츠를 빠르게 만들 수 있도록 도와줍니다.

어떤 특정한 맥락에서 "Autogen"에 대해 질문하신 것인지 더 구체적으로 말씀해 주시면, 더 자세한 정보를 제공해 드릴 수 있습니다.

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...
userproxyagent (to assistant):



--------------------------------------------------------------------------------
assistant (to userproxyagent):

안녕하세요! 질문이나 도움이 필요하신 부분이 있다면 말씀해 주세요. 어떤 정보를 찾고 계신가요?

--------------------------------------------------------------------------------
userproxyagent (to assistant):

AutoGen이 뭐야?

----------------------------------

KeyboardInterrupt: Interrupted by user

## 예제 15.8 OpenAI 임베딩 모델을 사용하도록 설정하기

In [ ]:
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
                api_key=openai_api_key,
                model_name="text-embedding-3-small"
            )

ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
        "embedding_function": openai_ef,
        "collection_name": "openai-embedding-3",
    },
)

assistant.reset()
ragproxyagent.initiate_chat(assistant, problem="Autogen이 뭐야?")

# assistant (to ragproxyagent):
# AutoGen은 여러 에이전트가 상호 대화하여 작업을 해결할 수 있는 LLM(Large Language Model) 애플리케이션 개발을 가능하게 하는 프레임워크입니다. AutoGen 에이전트는 사용자 정의 가능하며, 대화 가능하고, 인간 참여를 원활하게 허용합니다. LLM, 인간 입력, 도구의 조합을 사용하는 다양한 모드에서 작동할 수 있습니다.

## 예제 15.9 대화에 참여할 에이전트

In [ ]:
def termination_msg(x):
    return isinstance(x, dict) and "TERMINATE" == str(x.get("content", ""))[-9:].upper()

# RAG를 사용하지 않는 사용자 역할 에이전트
user = autogen.UserProxyAgent(
    name="Admin",
    is_termination_msg=termination_msg,
    human_input_mode="NEVER",
    system_message="The boss who ask questions and give tasks.",
    code_execution_config=False,
    default_auto_reply="Reply `TERMINATE` if the task is done.",
)
# RAG를 사용하는 사용자 역할 에이전트
user_rag = RetrieveUserProxyAgent(
    name="Admin_RAG",
    is_termination_msg=termination_msg,
    system_message="Assistant who has extra content retrieval power for solving difficult problems.",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config=False,
    retrieve_config={
        "task": "code",
        "docs_path": "https://raw.githubusercontent.com/microsoft/autogen/main/samples/apps/autogen-studio/README.md",
        "chunk_token_size": 1000,
        "collection_name": "groupchat-rag",
    }
)
# 프로그래머 역할의 에이전트
coder = AssistantAgent(
    name="Senior_Python_Engineer",
    is_termination_msg=termination_msg,
    system_message="You are a senior python engineer. Reply `TERMINATE` in the end when everything is done.",
    llm_config=llm_config,
)
# 프로덕트 매니저 역할의 에이전트
pm = autogen.AssistantAgent(
    name="Product_Manager",
    is_termination_msg=termination_msg,
    system_message="You are a product manager. Reply `TERMINATE` in the end when everything is done.",
    llm_config=llm_config,
)

PROBLEM = "AutoGen Studio는 무엇이고 AutoGen Studio로 어떤 제품을 만들 수 있을까?"

## 예제 15.10 RAG 사용 여부에 따른 2개의 그룹챗 정의 및 실행

In [9]:
def _reset_agents():
    user.reset()
    user_rag.reset()
    coder.reset()
    pm.reset()

def rag_chat():
    _reset_agents()
    groupchat = autogen.GroupChat(
        agents=[user_rag, coder, pm],
        messages=[], max_round=12, speaker_selection_method="round_robin"
    )
    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

    user_rag.initiate_chat(
        manager,
        problem=PROBLEM,
    )

def norag_chat():
    _reset_agents()
    groupchat = autogen.GroupChat(
        agents=[user, coder, pm],
        messages=[],
        max_round=12,
        speaker_selection_method="auto",
        allow_repeat_speaker=False,
    )
    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

    user.initiate_chat(
        manager,
        message=PROBLEM,
    )

## 예제 15.11 2개의 그룹챗을 실행한 결과 비교

In [10]:
norag_chat()
# AutoGen Studio는 자동화된 코드 생성 도구입니다. 이 도구를 사용하면 개발자들이 더 빠르게, 더 효율적으로 코드를 작성할 수 있습니다.
# AutoGen Studio를 사용하면 다양한 유형의 소프트웨어 제품을 만들 수 있습니다. 예를 들어, 웹 애플리케이션, 모바일 애플리케이션, 데스크톱 애플리케이션, API, 데이터베이스 등을 만들 수 있습니다.
# ...

NameError: name 'user' is not defined

In [11]:
rag_chat()
# AutoGen Studio는 AutoGen 프레임워크를 기반으로 한 AI 앱입니다. 이 앱은 AI 에이전트를 빠르게 프로토타입화하고, 스킬을 향상시키고, 워크플로우로 구성하고, 작업을 완료하기 위해 그들과 상호 작용하는 데 도움을 줍니다. 이 앱은 GitHub의 [microsoft/autogen](https://github.com/microsoft/autogen/tree/main/samples/apps/autogen-studio)에서 코드를 찾을 수 있습니다.
# AutoGen Studio를 사용하면 다음과 같은 기능을 수행할 수 있습니다:
# - 에이전트를 구축/구성하고, 그들의 구성(예: 스킬, 온도, 모델, 에이전트 시스템 메시지, 모델 등)을 수정하고, 워크플로우로 구성합니다.
# ...

NameError: name 'user' is not defined